# Constructing a Custom Neural Architecture — Starter Notebook

This is an **ungraded, self-paced lab**. Fill in each `TODO` section below, run the
notebook end to end, and compare your printed output against the **Expected Output
Reference** section at the very bottom. There's nothing to submit — this is a
self-check.

A fixed random seed (`torch.manual_seed(42)`) is used throughout so that, if every
section is implemented as described, your printed values should match the reference
values exactly.


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)


## Section 1: CustomNet.__init__() — layer and parameter registration

Complete the `__init__` method below:
- Replace the native Python list with an `nn.ModuleList` that builds `nn.Linear`
  layers based on the `hidden_sizes` argument (one `nn.Linear` between each
  consecutive pair of sizes in the list).
- Register a standalone learnable tensor called `output_bias` using `nn.Parameter`
  — a single-element tensor initialized to zero, added to the network's final
  output. This is not part of any `nn.Linear` layer, so it must be registered
  directly.
- Store `scale` as a plain attribute (not a parameter — this value is fixed, not
  learned).

**Why this matters:** layers stored in a plain Python list are invisible to
PyTorch's parameter registry. Only modules and tensors registered through
`nn.ModuleList`, `nn.ModuleDict`, or `nn.Parameter` will have their gradients
tracked and updated during training.


In [ ]:
class CustomNet(nn.Module):
    def __init__(self, hidden_sizes, scale=2.0):
        super().__init__()

        # TODO: replace this native Python list with an nn.ModuleList.
        # Build one nn.Linear layer between each consecutive pair of
        # sizes in hidden_sizes.
        self.layers = []  # <-- BROKEN: PyTorch cannot see layers stored here

        self.scale = scale

        # TODO: register a standalone learnable parameter called
        # output_bias — a single-element tensor initialized to zero —
        # using nn.Parameter.
        self.output_bias = None  # <-- TODO


In [ ]:
# Sanity check: once Section 1 is complete, this should print every
# nn.Linear weight/bias AND output_bias — nothing should be missing.
_test_model = CustomNet(hidden_sizes=[4, 8, 4, 1])
for name, _ in _test_model.named_parameters():
    print(name)


## Section 2: CustomActivation — custom autograd operation

Complete the `CustomActivation` class, which extends `torch.autograd.Function`.
This activation should behave like a **scaled ReLU**: positive inputs are
multiplied by `scale`, negative inputs become zero.

- In `forward()`: compute a mask of which elements of `x` are positive, save
  whatever tensors you'll need for `backward()` using `ctx.save_for_backward()`,
  and return `x * mask * scale`.
- In `backward()`: retrieve your saved tensors with `ctx.saved_tensors`, and
  return the gradient of the loss with respect to `x` — the incoming
  `grad_output`, scaled by the same mask and `scale` used in the forward pass.
  Remember: `backward()` must return one gradient per argument to `forward()`
  (here, `x` and `scale`) — return `None` for `scale`, since it isn't a
  learnable tensor.


In [ ]:
class CustomActivation(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, scale):
        # TODO: compute a mask of where x > 0
        # TODO: save whatever tensors backward() will need
        # TODO: return the scaled, masked output
        pass

    @staticmethod
    def backward(ctx, grad_output):
        # TODO: retrieve saved tensors
        # TODO: compute and return the gradient with respect to x
        # TODO: return None for the scale argument's gradient
        pass


## Section 3: CustomNet.forward() — wiring the layers together

Complete the `forward()` method:
- Pass `x` through each layer in `self.layers`, in order.
- Apply `CustomActivation.apply(x, self.scale)` after every layer **except the
  last one** (the final layer's output should NOT go through the activation).
- Add `self.output_bias` to the final result before returning it.

Run the cell below afterward with the provided synthetic input to confirm no
shape-mismatch errors occur.


In [ ]:
class CustomNet(CustomNet):  # reopen the class to add forward()
    def forward(self, x):
        # TODO: pass x through each layer in self.layers, applying
        # CustomActivation.apply(x, self.scale) after every layer
        # except the last one.
        # TODO: add self.output_bias to the final result before returning.
        pass


In [ ]:
# Shape check — run this once Sections 1-3 are complete.
# Expected shape: (2, 1). A mismatch here means your hidden_sizes wiring
# in Section 1 or your layer loop in Section 3 is misaligned.
_test_model = CustomNet(hidden_sizes=[4, 8, 4, 1])
_test_input = torch.randn(2, 4)
_test_output = _test_model(_test_input)
print("output shape:", tuple(_test_output.shape))


## Section 4: custom_loss() — manual loss computation

Implement `custom_loss(predictions, targets)` using plain tensor operations —
compute the mean squared error manually. Do **not** use `nn.MSELoss` or any
other pre-built loss module.


In [ ]:
def custom_loss(predictions, targets):
    # TODO: compute mean squared error manually using tensor operations
    # (no nn.MSELoss or other pre-built loss modules).
    pass


## Section 5: Execution cell — run the full pipeline

This cell instantiates the model, runs the synthetic input through it,
computes the loss, calls `loss.backward()`, and prints every registered
parameter's gradient. This is already written for you — once Sections 1-4
are complete, just run it.

**Before inspecting your printed values, review Step 6 below** on memory
boundaries — it affects how you should log or print tensor values in your
own future training loops, even though it doesn't change this cell's output.


In [ ]:
# Re-seed here so this cell's results are reproducible regardless of
# how many random draws the earlier sanity-check cells consumed.
torch.manual_seed(42)

hidden_sizes = [4, 8, 4, 1]
model = CustomNet(hidden_sizes=hidden_sizes, scale=2.0)

sample_input = torch.randn(2, 4)   # batch_size=2, input_size=4
target = torch.randn(2, 1)

output = model(sample_input)
loss = custom_loss(output, target)

loss.backward()

print("Output shape:", tuple(output.shape))
print("Loss value:", loss.item())
print()
for name, param in model.named_parameters():
    grad_shape = tuple(param.grad.shape) if param.grad is not None else None
    print(f"{name}: grad is None? {param.grad is None}  (shape: {grad_shape})")


## Step 6: Confirm you've respected memory boundaries

This step is a conceptual check, not additional code to submit here.

If you were logging this loss value across many training iterations (for
example, appending it to a list to plot later), you should **never** do this:

```python
loss_log.append(loss)  # retains the ENTIRE computational graph in memory
```

Instead, always detach and extract the scalar value first:

```python
loss_log.append(loss.detach().item())  # stores only the number
```

This doesn't change anything about today's single forward/backward pass —
but it's the habit this lab is meant to reinforce, connecting back to
`.detach()` and `torch.no_grad()` from earlier in this module.


---
## Expected Output Reference (self-check only)

With `torch.manual_seed(42)` set at the top of the notebook and every section
implemented exactly as described, your Section 5 output should match the
values below.

```
Output shape: (2, 1)
Loss value: 0.3658738...

output_bias: grad is None? False  (shape: (1,))
layers.0.weight: grad is None? False  (shape: (8, 4))
layers.0.bias: grad is None? False  (shape: (8,))
layers.1.weight: grad is None? False  (shape: (4, 8))
layers.1.bias: grad is None? False  (shape: (4,))
layers.2.weight: grad is None? False  (shape: (1, 4))
layers.2.bias: grad is None? False  (shape: (1,))
```

**If your shapes don't match:** revisit your `hidden_sizes` wiring in Section 1
or your layer loop in Section 3.

**If any `.grad` prints as `None`:** confirm the parameter is registered via
`nn.Parameter` or `nn.ModuleList` (Section 1), and that `loss.backward()` ran
without error before this check.

**If you want to compare exact gradient numbers, not just shapes:** the full
reference tensors are available from your instructor / lab administrator —
shape-matching alone is normally enough to confirm your implementation is
structurally correct.
